# Praktikum 3 - Image Classification

**Kelompok 5**  
Rayhan Hidayatul Fikri - 18223022  
Princessfa Azzahra A

Notebook ini dibersihkan ulang supaya seluruh proses memakai dataset lokal (`train.csv`, `val.csv`, `test.csv`, dan folder `images`) serta hanya menggunakan label `0-19` sesuai instruksi. Alur kerja dibuat eksplisit: data cleaning, preprocessing, feature engineering, data splitting, model CNN dari nol, pre-trained model, evaluasi, perbandingan, dan kesimpulan.


#### **1. Import Libraries**

In [1]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, UnidentifiedImageError

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, applications

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    top_k_accuracy_score,
)

sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow version:", tf.__version__)


ModuleNotFoundError: No module named 'numpy'

#### **2. Konfigurasi Path dan Parameter untuk Google Colab**

Cell ini dibuat agar notebook bisa jalan di Google Colab. Default-nya membaca file dari `/content`, yaitu lokasi upload manual Colab. Jika ingin memakai Google Drive, ubah `USE_GOOGLE_DRIVE = True` dan sesuaikan `DRIVE_PROJECT_DIR`.


In [ ]:
import zipfile

# Untuk Colab upload manual: biarkan False, lalu upload train.csv, val.csv, test.csv, dan images.zip ke /content.
# Untuk Google Drive: ubah True dan sesuaikan DRIVE_PROJECT_DIR.
USE_GOOGLE_DRIVE = False
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Prak3-AIForBizz")

IN_COLAB = Path("/content").exists()

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_DIR = DRIVE_PROJECT_DIR
elif IN_COLAB:
    ROOT_DIR = Path("/content")
else:
    ROOT_DIR = Path.cwd()

IMAGE_DIR = ROOT_DIR / "images"
IMAGE_ZIP = ROOT_DIR / "images.zip"
TRAIN_CSV = ROOT_DIR / "train.csv"
VAL_CSV = ROOT_DIR / "val.csv"
TEST_CSV = ROOT_DIR / "test.csv"

# Jika di Colab kamu upload images.zip, cell ini akan extract otomatis jika folder images belum ada.
if not IMAGE_DIR.exists() and IMAGE_ZIP.exists():
    try:
        with zipfile.ZipFile(IMAGE_ZIP, "r") as zip_ref:
            zip_ref.extractall(ROOT_DIR)
        print(f"Berhasil extract: {IMAGE_ZIP}")
    except zipfile.BadZipFile as exc:
        raise ValueError(
            "images.zip tidak valid/corrupt. Zip ulang folder images sebagai .zip asli, "
            "upload ulang ke Colab, lalu run cell ini lagi."
        ) from exc

# Antisipasi jika zip berisi folder project di dalamnya, misalnya Prak3-AIForBizz/images.
if not IMAGE_DIR.exists():
    candidate_dirs = [path for path in ROOT_DIR.rglob("images") if path.is_dir()]
    if candidate_dirs:
        IMAGE_DIR = candidate_dirs[0]

LABEL_MIN = 0
LABEL_MAX = 19
NUM_CLASSES = LABEL_MAX - LABEL_MIN + 1
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

print("ROOT_DIR:", ROOT_DIR)
print("IMAGE_DIR:", IMAGE_DIR)
print("Jumlah file gambar:", len(list(IMAGE_DIR.glob("*"))) if IMAGE_DIR.exists() else 0)

for file_path in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    if not file_path.exists():
        raise FileNotFoundError(f"File tidak ditemukan: {file_path}. Upload file CSV ke {ROOT_DIR}.")

if not IMAGE_DIR.exists():
    raise FileNotFoundError(
        f"Folder images tidak ditemukan di {ROOT_DIR}. Upload folder images atau images.zip, lalu run cell ini lagi."
    )


## 3. Load Metadata Dataset

CSV dibaca dari path yang sudah dikonfigurasi untuk Colab. Cell ini juga menampilkan isi folder kerja supaya mudah mengecek apakah file sudah ter-upload ke lokasi yang benar.


In [ ]:
print("Isi ROOT_DIR:")
for item in sorted(ROOT_DIR.iterdir()):
    print("-", item.name)

def read_metadata(csv_path: Path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)
    print(f"{csv_path.name}: {df.shape[0]} baris, {df.shape[1]} kolom")
    return df

raw_train_df = read_metadata(TRAIN_CSV)
raw_val_df = read_metadata(VAL_CSV)
raw_test_df = read_metadata(TEST_CSV)

display(raw_train_df.head())


## 4. Data Cleaning: Missing Values, Duplicates, dan Structural Error

Cleaning dilakukan pada metadata, bukan mengubah file gambar asli. Langkah yang dilakukan:

- Menstandarkan nama kolom menjadi `filename`, `class_name`, dan `label`.
- Menghapus spasi/format tidak konsisten pada `filename` dan `class_name`.
- Mengubah label menjadi numerik.
- Membuang baris yang missing pada kolom penting.
- Menghapus duplikat metadata dan duplikat filename.
- Memastikan hanya label `0-19` yang dipakai.
- Memastikan file gambar benar-benar ada di folder `images`.


In [ ]:
def clean_metadata(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    cleaned = df.copy()
    cleaned.columns = cleaned.columns.str.strip().str.lower()
    cleaned = cleaned.rename(columns={"classes": "class_name", "labels": "label"})

    required_cols = ["filename", "class_name", "label"]
    missing_cols = [col for col in required_cols if col not in cleaned.columns]
    if missing_cols:
        raise ValueError(f"Kolom wajib tidak ada di {split_name}: {missing_cols}")

    cleaned["filename"] = cleaned["filename"].astype(str).str.strip()
    cleaned["class_name"] = (
        cleaned["class_name"]
        .astype(str)
        .str.strip()
        .str.upper()
        .str.replace(r"\s+", "", regex=True)
    )
    cleaned["label"] = pd.to_numeric(cleaned["label"], errors="coerce")

    report = {
        "split": split_name,
        "initial_rows": len(cleaned),
        "missing_rows": int(cleaned[required_cols].isna().any(axis=1).sum()),
        "duplicate_rows": int(cleaned.duplicated().sum()),
        "duplicate_filenames": int(cleaned.duplicated(subset=["filename"]).sum()),
    }

    cleaned = cleaned.dropna(subset=required_cols)
    cleaned["label"] = cleaned["label"].astype(int)
    cleaned = cleaned[(cleaned["label"] >= LABEL_MIN) & (cleaned["label"] <= LABEL_MAX)]
    cleaned = cleaned.drop_duplicates()
    cleaned = cleaned.drop_duplicates(subset=["filename"], keep="first")

    cleaned["image_path"] = cleaned["filename"].apply(lambda name: str(IMAGE_DIR / name))
    cleaned["file_exists"] = cleaned["image_path"].apply(lambda p: Path(p).exists())
    report["missing_image_files"] = int((~cleaned["file_exists"]).sum())
    cleaned = cleaned[cleaned["file_exists"]].drop(columns="file_exists")
    cleaned["split"] = split_name

    return cleaned.reset_index(drop=True), report

train_df, train_clean_report = clean_metadata(raw_train_df, "train")
val_df, val_clean_report = clean_metadata(raw_val_df, "val")
test_df, test_clean_report = clean_metadata(raw_test_df, "test")

cleaning_report = pd.DataFrame([train_clean_report, val_clean_report, test_clean_report])
cleaning_report


## 5. Cek Data Leakage Antar Split

Karena split sudah disediakan, split tersebut dipertahankan. Namun filename yang muncul di lebih dari satu split perlu dibuang dari split evaluasi agar performa tidak bias.


In [ ]:
train_files = set(train_df["filename"])
val_files = set(val_df["filename"])
test_files = set(test_df["filename"])

leak_train_val = train_files & val_files
leak_train_test = train_files & test_files
leak_val_test = val_files & test_files

print("Overlap train-val:", len(leak_train_val))
print("Overlap train-test:", len(leak_train_test))
print("Overlap val-test:", len(leak_val_test))

if leak_train_val or leak_train_test or leak_val_test:
    val_df = val_df[~val_df["filename"].isin(train_files)].reset_index(drop=True)
    test_df = test_df[~test_df["filename"].isin(train_files | set(val_df["filename"]))].reset_index(drop=True)
    print("Leakage ditemukan dan sudah dibersihkan dari validation/test.")
else:
    print("Tidak ada data leakage berdasarkan filename.")


## 6. Feature Engineering Metadata Gambar

Feature engineering yang relevan untuk image classification dilakukan secara surgical: metadata gambar dipakai untuk audit kualitas data dan memilih preprocessing yang tepat, bukan menambah fitur tabular ke CNN. Fitur yang dibuat:

- `width` dan `height`: ukuran asli gambar.
- `aspect_ratio`: rasio lebar terhadap tinggi.
- `megapixels`: ukuran gambar dalam megapixel.
- `file_size_kb`: ukuran file.

Fitur ini membantu mendeteksi outlier ukuran/aspect ratio dan memastikan treatment preprocessing tidak merusak bentuk pesawat.


In [ ]:
def extract_image_features(df: pd.DataFrame) -> pd.DataFrame:
    enriched = df.copy()
    widths, heights, modes, file_sizes, valid_images = [], [], [], [], []

    for path in enriched["image_path"]:
        image_path = Path(path)
        file_sizes.append(image_path.stat().st_size / 1024)
        try:
            with Image.open(image_path) as img:
                widths.append(img.width)
                heights.append(img.height)
                modes.append(img.mode)
                valid_images.append(True)
        except (UnidentifiedImageError, OSError):
            widths.append(np.nan)
            heights.append(np.nan)
            modes.append("invalid")
            valid_images.append(False)

    enriched["width"] = widths
    enriched["height"] = heights
    enriched["mode"] = modes
    enriched["file_size_kb"] = file_sizes
    enriched["is_valid_image"] = valid_images
    enriched["aspect_ratio"] = enriched["width"] / enriched["height"]
    enriched["megapixels"] = (enriched["width"] * enriched["height"]) / 1_000_000
    return enriched

train_df = extract_image_features(train_df)
val_df = extract_image_features(val_df)
test_df = extract_image_features(test_df)

image_quality_report = pd.concat([train_df, val_df, test_df]).groupby("split").agg(
    rows=("filename", "count"),
    valid_images=("is_valid_image", "sum"),
    min_width=("width", "min"),
    max_width=("width", "max"),
    min_height=("height", "min"),
    max_height=("height", "max"),
    median_aspect_ratio=("aspect_ratio", "median"),
    median_megapixels=("megapixels", "median"),
)
image_quality_report


## 7. Outlier Detection dan Treatment

Untuk gambar, outlier ukuran tidak langsung dihapus karena ukuran asli yang berbeda adalah hal normal. Metode yang dipakai:

- Deteksi outlier memakai IQR pada `aspect_ratio` dan `megapixels`.
- Gambar invalid dibuang.
- Outlier ukuran diperlakukan saat preprocessing dengan `resize_with_pad`, sehingga rasio bentuk pesawat dipertahankan dan gambar menjadi ukuran seragam `224x224`.

Ini lebih aman daripada cropping agresif karena pesawat bisa terpotong dan informasi kelas hilang.


In [ ]:
def add_outlier_flags(reference_df: pd.DataFrame, target_df: pd.DataFrame, cols=("aspect_ratio", "megapixels")) -> pd.DataFrame:
    flagged = target_df.copy()
    flagged["is_size_outlier"] = False

    for col in cols:
        q1 = reference_df[col].quantile(0.25)
        q3 = reference_df[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        col_flag = f"{col}_outlier"
        flagged[col_flag] = ~flagged[col].between(lower, upper)
        flagged["is_size_outlier"] = flagged["is_size_outlier"] | flagged[col_flag]

    return flagged

train_df = add_outlier_flags(train_df, train_df)
val_df = add_outlier_flags(train_df, val_df)
test_df = add_outlier_flags(train_df, test_df)

# Drop only invalid/corrupt images. Size outliers are kept and handled by resize_with_pad.
train_df = train_df[train_df["is_valid_image"]].reset_index(drop=True)
val_df = val_df[val_df["is_valid_image"]].reset_index(drop=True)
test_df = test_df[test_df["is_valid_image"]].reset_index(drop=True)

pd.concat([train_df, val_df, test_df]).groupby("split")["is_size_outlier"].agg(["sum", "count", "mean"])


## 8. EDA: Distribusi Kelas dan Ukuran Gambar

Distribusi kelas perlu dicek karena imbalance dapat membuat akurasi terlihat tinggi tetapi performa per kelas buruk. Pada dataset ini, kelas relatif seimbang sehingga metrik macro-F1 tetap relevan untuk melihat kualitas rata-rata antar kelas.


In [ ]:
label_to_class = (
    pd.concat([train_df, val_df, test_df])[["label", "class_name"]]
    .drop_duplicates()
    .sort_values("label")
    .set_index("label")["class_name"]
    .to_dict()
)
class_names = [label_to_class[i] for i in range(NUM_CLASSES)]

fig, axes = plt.subplots(1, 3, figsize=(18, 4), sharey=True)
for ax, df, title in zip(axes, [train_df, val_df, test_df], ["Train", "Validation", "Test"]):
    sns.countplot(data=df, x="label", ax=ax, color="#4C78A8")
    ax.set_title(f"Distribusi Label - {title}")
    ax.set_xlabel("Label")
    ax.set_ylabel("Jumlah Data")
    ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(pd.concat([train_df, val_df, test_df]), x="aspect_ratio", hue="split", kde=True, ax=axes[0])
sns.histplot(pd.concat([train_df, val_df, test_df]), x="megapixels", hue="split", kde=True, ax=axes[1])
axes[0].set_title("Distribusi Aspect Ratio")
axes[1].set_title("Distribusi Megapixels")
plt.tight_layout()
plt.show()


## 9. Visualisasi Sampel Gambar

Visualisasi ini membantu memastikan label dan gambar sesuai sebelum model dilatih.


In [ ]:
def show_samples_per_class(df: pd.DataFrame, samples_per_class: int = 3):
    labels = sorted(df["label"].unique())
    fig, axes = plt.subplots(len(labels), samples_per_class, figsize=(3 * samples_per_class, 2.1 * len(labels)))

    for row_idx, label in enumerate(labels):
        sample_df = df[df["label"] == label].sample(
            n=min(samples_per_class, (df["label"] == label).sum()),
            random_state=SEED,
        )
        for col_idx, (_, row) in enumerate(sample_df.iterrows()):
            ax = axes[row_idx, col_idx] if len(labels) > 1 else axes[col_idx]
            img = Image.open(row["image_path"]).convert("RGB")
            ax.imshow(img)
            ax.set_title(f"{label}: {row['class_name']}", fontsize=9)
            ax.axis("off")

        for col_idx in range(len(sample_df), samples_per_class):
            ax = axes[row_idx, col_idx] if len(labels) > 1 else axes[col_idx]
            ax.axis("off")

    plt.tight_layout()
    plt.show()

show_samples_per_class(train_df, samples_per_class=3)


## 10. Data Preprocessing dan Feature Scaling

Preprocessing utama untuk CNN:

- Decode JPG sebagai RGB 3-channel untuk memperbaiki structural inconsistency mode warna.
- `resize_with_pad` ke `224x224` supaya semua input seragam tanpa memotong pesawat.
- Scaling pixel dari `0-255` menjadi `0-1`.
- Normalization layer diadaptasi dari train set untuk model CNN dari nol.
- Data augmentation sebagai feature engineering visual: rotasi kecil, translasi, zoom, contrast, dan horizontal flip. Augmentasi dibuat moderat agar tidak mengubah kelas pesawat.


In [ ]:
def make_tf_dataset(df: pd.DataFrame, shuffle: bool = False) -> tf.data.Dataset:
    paths = df["image_path"].astype(str).values
    labels = df["label"].astype("int32").values
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    def load_and_preprocess(path, label):
        image = tf.io.read_file(path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize_with_pad(image, IMG_SIZE[0], IMG_SIZE[1])
        image = tf.cast(image, tf.float32) / 255.0
        return image, label

    ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(buffer_size=len(df), seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_tf_dataset(train_df, shuffle=True)
val_ds = make_tf_dataset(val_df, shuffle=False)
test_ds = make_tf_dataset(test_df, shuffle=False)

normalizer = layers.Normalization(axis=-1, name="train_pixel_normalization")
normalizer.adapt(train_ds.map(lambda images, labels: images))

for images, labels in train_ds.take(1):
    print("Batch image shape:", images.shape)
    print("Batch label shape:", labels.shape)
    print("Pixel range:", float(tf.reduce_min(images)), "to", float(tf.reduce_max(images)))


## 11. Data Splitting

Dataset menggunakan split bawaan:

- `train.csv` untuk training.
- `val.csv` untuk validasi selama training dan tuning.
- `test.csv` hanya untuk evaluasi akhir.

Split bawaan dipertahankan karena sudah seimbang dan tidak ada overlap filename antar split.


In [ ]:
split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "num_classes": [train_df["label"].nunique(), val_df["label"].nunique(), test_df["label"].nunique()],
})
split_summary


## 12. Model 1: CNN dari Nol

Arsitektur CNN dibuat cukup kuat untuk dataset kecil-menengah:

- Blok Conv2D bertingkat untuk menangkap pola dari edge sederhana sampai bentuk pesawat.
- Batch Normalization untuk stabilitas training.
- MaxPooling untuk reduksi dimensi spasial.
- Dropout dan L2 regularization untuk mengurangi overfitting.
- GlobalAveragePooling2D agar parameter lebih sedikit daripada Flatten besar.
- Softmax 20 kelas untuk label `0-19`.


In [ ]:
data_augmentation = models.Sequential(
    [
        layers.RandomFlip("horizontal", seed=SEED),
        layers.RandomRotation(0.04, fill_mode="reflect", seed=SEED),
        layers.RandomTranslation(0.06, 0.06, fill_mode="reflect", seed=SEED),
        layers.RandomZoom(0.08, fill_mode="reflect", seed=SEED),
        layers.RandomContrast(0.12, seed=SEED),
    ],
    name="visual_feature_engineering",
)


def build_cnn_model(input_shape=(224, 224, 3), num_classes=20):
    weight_decay = 1e-4
    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = normalizer(x)

    for filters, dropout_rate in [(32, 0.10), (64, 0.15), (128, 0.20), (256, 0.25)]:
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_regularizer=tf.keras.regularizers.l2(weight_decay))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", use_bias=False, kernel_regularizer=tf.keras.regularizers.l2(weight_decay))(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D()(x)
        x = layers.Dropout(dropout_rate)(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(weight_decay))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.40)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="cnn_from_scratch")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy")],
    )
    return model

cnn_model = build_cnn_model(num_classes=NUM_CLASSES)
cnn_model.summary()


## 13. Training Model CNN

Callback dipakai agar training berhenti saat validasi tidak membaik dan learning rate otomatis turun saat model mulai plateau. Model terbaik disimpan berdasarkan `val_accuracy`.


In [ ]:
cnn_callbacks = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=8, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint("best_cnn_from_scratch.keras", monitor="val_accuracy", save_best_only=True, verbose=1),
]

EPOCHS_CNN = 40

history_cnn = cnn_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_CNN,
    callbacks=cnn_callbacks,
)


## 14. Model 2: Pre-trained Model

Model pembanding memakai MobileNetV2 yang sudah dilatih di ImageNet. Strateginya transfer learning:

- Base model dibekukan pada tahap awal agar feature extractor stabil.
- Classification head baru dilatih untuk 20 kelas pesawat.
- Setelah itu, sebagian layer atas dapat di-unfreeze untuk fine-tuning ringan.

Catatan: pertama kali menjalankan `weights="imagenet"` mungkin membutuhkan internet untuk mengunduh bobot.


In [ ]:
def build_transfer_model(input_shape=(224, 224, 3), num_classes=20):
    try:
        base_model = applications.MobileNetV2(
            input_shape=input_shape,
            include_top=False,
            weights="imagenet",
        )
        print("Menggunakan bobot ImageNet.")
    except Exception as exc:
        print("Bobot ImageNet gagal dimuat. Model dibuat tanpa pre-trained weights.")
        print("Alasan:", exc)
        base_model = applications.MobileNetV2(
            input_shape=input_shape,
            include_top=False,
            weights=None,
        )

    base_model.trainable = False

    inputs = layers.Input(shape=input_shape)
    x = data_augmentation(inputs)
    x = layers.Lambda(lambda image: applications.mobilenet_v2.preprocess_input(image * 255.0), name="mobilenet_preprocess")(x)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)

    model = models.Model(inputs, outputs, name="mobilenetv2_transfer")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=3e-4),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy")],
    )
    return model, base_model

transfer_model, transfer_base = build_transfer_model(num_classes=NUM_CLASSES)
transfer_model.summary()


## 15. Training Pre-trained Model

Training dilakukan dua tahap. Tahap pertama melatih classification head. Tahap kedua melakukan fine-tuning ringan pada layer atas MobileNetV2 dengan learning rate kecil.


In [ ]:
transfer_callbacks = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=6, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2, min_lr=1e-7, verbose=1),
    callbacks.ModelCheckpoint("best_mobilenetv2_transfer.keras", monitor="val_accuracy", save_best_only=True, verbose=1),
]

EPOCHS_TRANSFER_HEAD = 20

history_transfer_head = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_TRANSFER_HEAD,
    callbacks=transfer_callbacks,
)


In [ ]:
# Fine-tuning ringan: buka sebagian layer teratas, tetapi BatchNorm tetap dibekukan agar training stabil.
transfer_base.trainable = True
for layer in transfer_base.layers[:-30]:
    layer.trainable = False
for layer in transfer_base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

transfer_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy", tf.keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3_accuracy")],
)

EPOCHS_TRANSFER_FINE_TUNE = 15

history_transfer_finetune = transfer_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_TRANSFER_FINE_TUNE,
    callbacks=transfer_callbacks,
)


## 16. Visualisasi Learning Curve

Learning curve membantu membaca overfitting dan underfitting:

- Jika train accuracy naik tetapi validation accuracy stagnan/turun, model overfitting.
- Jika train dan validation sama-sama rendah, model underfitting.
- Model yang bagus biasanya punya validation loss turun stabil dan gap train-validation tidak terlalu jauh.


In [ ]:
def merge_histories(*histories):
    merged = {}
    for history in histories:
        if history is None:
            continue
        for key, values in history.history.items():
            merged.setdefault(key, []).extend(values)
    return merged


def plot_learning_curves(history_dict, title):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    axes[0].plot(history_dict.get("accuracy", []), label="Train Accuracy")
    axes[0].plot(history_dict.get("val_accuracy", []), label="Validation Accuracy")
    axes[0].set_title(f"{title} - Accuracy")
    axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Accuracy")
    axes[0].legend()

    axes[1].plot(history_dict.get("loss", []), label="Train Loss")
    axes[1].plot(history_dict.get("val_loss", []), label="Validation Loss")
    axes[1].set_title(f"{title} - Loss")
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Loss")
    axes[1].legend()

    plt.tight_layout()
    plt.show()

plot_learning_curves(history_cnn.history, "CNN dari Nol")
transfer_history = merge_histories(history_transfer_head, history_transfer_finetune)
plot_learning_curves(transfer_history, "MobileNetV2 Transfer Learning")


## 17. Pengujian Performa Model

Sel ini adalah pengujian performa utama. Metrik yang dipakai:

- `accuracy`: proporsi prediksi benar secara total.
- `balanced_accuracy`: rata-rata recall setiap kelas, lebih adil bila kelas tidak seimbang.
- `macro_f1`: rata-rata F1 semua kelas, penting untuk melihat performa antar kelas.
- `top3_accuracy`: apakah label benar masuk 3 probabilitas tertinggi, berguna untuk klasifikasi visual yang mirip.
- `classification_report`: precision, recall, dan F1 per kelas.
- `confusion_matrix`: kelas mana yang sering tertukar.


In [ ]:
def predict_dataset(model, dataset):
    y_true_batches = []
    y_prob_batches = []

    for images, labels in dataset:
        y_true_batches.append(labels.numpy())
        y_prob_batches.append(model.predict(images, verbose=0))

    y_true = np.concatenate(y_true_batches)
    y_prob = np.concatenate(y_prob_batches)
    y_pred = np.argmax(y_prob, axis=1)
    return y_true, y_pred, y_prob


def evaluate_model(model, dataset, model_name, class_names):
    y_true, y_pred, y_prob = predict_dataset(model, dataset)

    metrics = {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
        "top3_accuracy": top_k_accuracy_score(y_true, y_prob, k=3, labels=np.arange(NUM_CLASSES)),
    }

    print(f"=== {model_name} ===")
    print(pd.Series(metrics).drop("model").to_string())
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"Confusion Matrix - {model_name}")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks(rotation=90)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    report_df = pd.DataFrame(classification_report(y_true, y_pred, target_names=class_names, output_dict=True)).T
    return metrics, report_df, (y_true, y_pred, y_prob)

cnn_metrics, cnn_report_df, cnn_predictions = evaluate_model(cnn_model, test_ds, "CNN dari Nol", class_names)
transfer_metrics, transfer_report_df, transfer_predictions = evaluate_model(transfer_model, test_ds, "MobileNetV2 Transfer Learning", class_names)

comparison_df = pd.DataFrame([cnn_metrics, transfer_metrics]).sort_values("macro_f1", ascending=False)
comparison_df


## 18. Analisis Kelas Paling Sulit

Bagian ini menampilkan kelas dengan F1 terendah. Kelas-kelas tersebut biasanya perlu dicek lagi gambarnya: bisa karena visual pesawat mirip, jumlah data terbatas, atau augmentasi belum cukup representatif.


In [ ]:
def show_worst_classes(report_df, model_name, n=5):
    per_class = report_df.loc[class_names, ["precision", "recall", "f1-score", "support"]].copy()
    per_class = per_class.sort_values("f1-score", ascending=True).head(n)
    print(f"Kelas tersulit - {model_name}")
    display(per_class)
    return per_class

worst_cnn = show_worst_classes(cnn_report_df, "CNN dari Nol")
worst_transfer = show_worst_classes(transfer_report_df, "MobileNetV2 Transfer Learning")


## 19. Visualisasi Prediksi Benar dan Salah

Grid ini membantu melakukan error analysis secara visual. Jika kesalahan banyak terjadi pada kelas pesawat yang bentuknya mirip, performa rendah bisa disebabkan fine-grained classification yang memang sulit.


In [ ]:
def show_prediction_examples(df, predictions, title, only_wrong=True, max_images=12):
    y_true, y_pred, y_prob = predictions
    result_df = df.copy().reset_index(drop=True)
    result_df["y_true"] = y_true
    result_df["y_pred"] = y_pred
    result_df["confidence"] = y_prob.max(axis=1)
    result_df["is_correct"] = result_df["y_true"] == result_df["y_pred"]

    if only_wrong:
        result_df = result_df[~result_df["is_correct"]]

    if result_df.empty:
        print("Tidak ada contoh yang sesuai filter.")
        return

    sample_df = result_df.sample(n=min(max_images, len(result_df)), random_state=SEED)
    cols = 4
    rows = int(np.ceil(len(sample_df) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(14, 3.5 * rows))
    axes = np.array(axes).reshape(-1)

    for ax, (_, row) in zip(axes, sample_df.iterrows()):
        img = Image.open(row["image_path"]).convert("RGB")
        ax.imshow(img)
        true_name = class_names[int(row["y_true"])]
        pred_name = class_names[int(row["y_pred"])]
        ax.set_title(f"True: {true_name}\nPred: {pred_name}\nConf: {row['confidence']:.2f}", fontsize=9)
        ax.axis("off")

    for ax in axes[len(sample_df):]:
        ax.axis("off")

    fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.show()

show_prediction_examples(test_df, cnn_predictions, "Contoh Salah Prediksi - CNN dari Nol", only_wrong=True)
show_prediction_examples(test_df, transfer_predictions, "Contoh Salah Prediksi - MobileNetV2", only_wrong=True)


## 20. Cara Menilai Performa CNN Sudah Bagus atau Belum

Gunakan patokan berikut saat membaca hasil evaluasi:

- **Bandingkan dengan baseline acak.** Karena ada 20 kelas, tebakan acak kira-kira hanya `1/20 = 5%` accuracy. Model harus jauh di atas angka ini.
- **Utamakan validation/test, bukan train.** Train accuracy tinggi tetapi validation/test rendah berarti overfitting.
- **Perhatikan macro-F1.** Untuk tugas multi-kelas, model lebih bagus jika macro-F1 tinggi dan tidak hanya bagus pada beberapa kelas.
- **Cek gap train-validation.** Gap accuracy lebih dari sekitar 10-15 poin biasanya tanda model mulai overfit.
- **Cek confusion matrix.** Jika beberapa kelas selalu tertukar, dataset mungkin butuh augmentasi tambahan, resolusi lebih tinggi, atau pre-trained model.
- **Bandingkan CNN vs pre-trained.** Jika MobileNetV2 jauh lebih baik, berarti dataset membutuhkan fitur visual yang lebih kuat daripada CNN kecil dari nol.

Sebagai aturan praktis untuk tugas ini: CNN mulai layak bila test accuracy dan macro-F1 jauh di atas baseline acak dan learning curve stabil. Model bisa dianggap bagus bila test accuracy/macro-F1 konsisten tinggi, confusion matrix cukup diagonal, dan performanya tidak jatuh jauh dari validation score.


In [ ]:
# Ringkasan otomatis untuk membantu membaca apakah CNN sudah cukup bagus.
random_baseline = 1 / NUM_CLASSES
cnn_test_acc = cnn_metrics["accuracy"]
cnn_macro_f1 = cnn_metrics["macro_f1"]
transfer_test_acc = transfer_metrics["accuracy"]

print(f"Baseline acak 20 kelas: {random_baseline:.2%}")
print(f"CNN test accuracy: {cnn_test_acc:.2%}")
print(f"CNN macro-F1: {cnn_macro_f1:.2%}")
print(f"MobileNetV2 test accuracy: {transfer_test_acc:.2%}")

if cnn_test_acc < random_baseline * 3:
    print("CNN belum bagus: performanya masih terlalu dekat dengan baseline acak.")
elif cnn_macro_f1 < 0.50:
    print("CNN mulai belajar, tetapi belum kuat: macro-F1 masih rendah untuk klasifikasi 20 kelas.")
elif transfer_test_acc - cnn_test_acc > 0.10:
    print("CNN cukup, tetapi pre-trained model jauh lebih baik. Untuk hasil akhir, MobileNetV2 lebih direkomendasikan.")
else:
    print("CNN sudah cukup bagus secara praktis: performa jauh di atas baseline dan kompetitif terhadap model pre-trained.")


## 21. Kesimpulan

Notebook ini sudah memakai dataset lokal dan label `0-19`. Data cleaning menunjukkan metadata, file gambar, dan split dapat divalidasi sebelum training. Treatment outlier dilakukan dengan strategi yang sesuai untuk gambar, yaitu menjaga semua gambar valid dan memakai `resize_with_pad` agar bentuk pesawat tidak terdistorsi. Dua model dibuat sesuai instruksi: CNN dari nol dan MobileNetV2 pre-trained. Model terbaik ditentukan dari metrik test set, terutama macro-F1, balanced accuracy, top-3 accuracy, dan confusion matrix.
